In [11]:
import csv
import json

class Relacion:
    def __init__(self, nodos, matriz):
        self.nodos = nodos
        self.matriz = matriz

    @classmethod
    def desde_csv(cls, filename):
        with open(filename, newline='') as f:
            reader = csv.reader(f)
            matriz = [list(map(int, row)) for row in reader]
        nodos = [str(i) for i in range(len(matriz))]
        return cls(nodos, matriz)

    @classmethod
    def desde_json(cls, filename):
        with open(filename) as f:
            estructura = json.load(f)
        nodos = estructura["P"]
        n = len(nodos)
        matriz = [[0]*n for _ in range(n)]
        for u, vecinos in estructura["E"].items():
            try:
                i = nodos.index(u)
                for v in vecinos:
                    try:
                        j = nodos.index(v)
                        matriz[i][j] = 1
                    except ValueError:
                        print(f"Advertencia: El nodo destino '{v}' de la relación '{u}' no se encuentra en la lista de nodos 'P'.")
            except ValueError:
                print(f"Advertencia: El nodo origen '{u}' de una relación no se encuentra en la lista de nodos 'P'.")
        return cls(nodos, matriz)


    def es_reflexiva(self):
        if not self.nodos:
            return True
        return all(self.matriz[i][i] == 1 for i in range(len(self.nodos)))

    def es_simetrica(self):
        n = len(self.nodos)
        return all(self.matriz[i][j] == self.matriz[j][i] for i in range(n) for j in range(n))

    def es_antisimetrica(self):
        n = len(self.nodos)
        for i in range(n):
            for j in range(n):
                if i != j and self.matriz[i][j] == 1 and self.matriz[j][i] == 1:
                    return False
        return True

    def es_transitiva(self):
        n = len(self.nodos)
        for i in range(n):
            for j in range(n):
                if self.matriz[i][j] == 1:
                    for k in range(n):
                        if self.matriz[j][k] == 1 and self.matriz[i][k] == 0:
                            return False
        return True

    def es_conexa(self):
        n = len(self.nodos)
        if n <= 1:
            return True
        for i in range(n):
            for j in range(n):
                if i != j and (self.matriz[i][j] == 1 or self.matriz[j][i] == 1):
                    return True
        return False


    def clasificar(self):
        reflex = self.es_reflexiva()
        sim = self.es_simetrica()
        anti = self.es_antisimetrica()
        trans = self.es_transitiva()


        print(f"¿Es reflexiva? {reflex}")
        print(f"¿Es simétrica? {sim}")
        print(f"¿Es antisimétrica? {anti}")
        print(f"¿Es transitiva? {trans}")

        if reflex and anti and trans:
             return "Orden total" if self.es_conexa() else "Orden parcial"
        elif reflex and sim and trans:
            return "Relación de equivalencia"
        else:
            return "No es ni orden ni equivalencia"


if __name__ == "__main__":
    r_json = Relacion.desde_json("01.json")
    print("Clasificación:", r_json.clasificar())

¿Es reflexiva? True
¿Es simétrica? False
¿Es antisimétrica? True
¿Es transitiva? True
Clasificación: Orden total
